## 텍스트 전처리

### 필요한 모듈 임포트

In [ ]:
# 필요한 모듈을 임포트합니다.
import os
import numpy as np
import pandas as pd
import ko_spelling_et5

In [ ]:
# 현재 작업 경로를 확인합니다.
os.getcwd()

In [ ]:
# data 폴더로 작업 경로를 변경합니다.
os.chdir(path='../data')

In [ ]:
# 현재 작업 경로에 있는 폴더명과 파일명을 확인합니다.
sorted(os.listdir())

### 실습 데이터셋 준비

In [ ]:
# xlsx 파일을 읽고 데이터프레임을 생성합니다.
df = pd.read_excel(io='Galaxy_Book_Review_20250131.xlsx')

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

In [ ]:
# df의 정보를 확인합니다.
df.info()

In [ ]:
# 연도별 리뷰 건수를 확인합니다.
df['date'].dt.year.value_counts().sort_index()

### 불필요한 문자/기호 제거 및 알파벳 통일

In [ ]:
# 정규 표현식 패턴을 설정합니다.
pattern = '[^가-힣A-Za-z0-9]+'

In [ ]:
# text 열에서 정규 표현식 패턴을 한 칸 공백으로 변경하고 text 열에 재할당합니다.
df['text'] = df['text'].str.replace(pat=pattern, repl=' ', regex=True)

In [ ]:
# text 열에서 모든 알파벳을 대문자로 변경하고 text 열에 재할당합니다.
df['text'] = df['text'].str.upper()

In [ ]:
# df의 처음 5행을 확인합니다.
df.head()

### 한글 맞춤법 검사

In [ ]:
# 하나의 텍스트로 한글 맞춤법 검사를 수행하고 corrected에 할당합니다.
corrected = ko_spelling_et5.correct(text='복리후생으로워라벨부터챙기자')

In [ ]:
# corrected(한글 맞춤법 검사 결과)를 확인합니다.
corrected

In [ ]:
# 한글 맞춤법 검사 대상인 여러 텍스트를 리스트로 생성합니다.
texts = ['지인짜 감사하빈다', '빠른 배송조아여', '증말 마시따', '고나리자에게 문의']

In [ ]:
# 여러 텍스트로 한글 맞춤법 검사를 수행한 결과를 확인합니다.
# [참고] text 매개변수에 문자열을 원소로 갖는 리스트, 배열, 시리즈를 지정하면
# 지정하면 한글 맞춤법 검사를 일괄 수행합니다.
ko_spelling_et5.correct(text=texts)

In [ ]:
# 문서 집합의 처음 10개 텍스트로 한글 맞춤법 검사를 수행한 결과를 확인합니다.
ko_spelling_et5.correct(text=df['text'].head(n=10))

In [ ]:
# 한글 맞춤법 검사 전 원본 문서와 비교합니다.
df['text'].head(n=10)

In [ ]:
# 한글 맞춤법 교정을 완료한 xlsx 파일을 읽고 데이터프레임을 생성합니다.
df = pd.read_excel(io='Galaxy_Book_Review_20250131_Corrected.xlsx')

In [ ]:
# df의 처음 10행을 확인합니다.
df.head(n=10)

### 형태소 분석 실행

In [ ]:
# 필요한 모듈을 임포트합니다.
from kiwipiepy import Kiwi

In [ ]:
# 한글 형태소 분석기 객체를 생성합니다.
kiwi = Kiwi(model_type='cong')

In [ ]:
# corrected(한글 맞춤법 검사 결과)로 형태소 분석을 실행한 결과를 확인합니다.
kiwi.tokenize(text=corrected, typos='basic')

### 불용어 삭제

In [ ]:
# 필요한 모듈을 임포트합니다.
from kiwipiepy.utils import Stopwords

In [ ]:
# 내장 불용어 객체를 생성합니다.
stopwords = Stopwords()

In [ ]:
# 내장 불용어 목록을 확인합니다.
stopwords.stopwords

In [ ]:
# 형태소 분석을 실행하고 불용어를 삭제한 결과를 확인합니다.
kiwi.tokenize(text=corrected, typos='basic', stopwords=stopwords)

### 사용자 사전 추가

In [ ]:
# 사용자 사전에 새로운 단어를 추가합니다.
kiwi.add_user_word(word='복리후생', tag='NNG')

In [ ]:
# 사용자 사전을 적용하여 형태소 분석을 실행한 결과를 tokens에 할당합니다.
tokens = kiwi.tokenize(text=corrected, typos='basic', stopwords=stopwords)

In [ ]:
# tokens(형태소 분석 결과)를 확인합니다.
tokens

In [ ]:
# 첫 번째 형태소의 단어와 품사를 확인합니다.
tokens[0].form, tokens[0].tag

### 품사 선택

In [ ]:
# 형태소 분석 결과에서 선택할 품사(용언과 체언) 목록을 리스트로 생성합니다.
verb_tags = ['VV', 'VV-R', 'VV-I', 'VA', 'VA-R', 'VA-I']
noun_tags = ['NNG', 'NNP']

In [ ]:
# 품사가 용언과 체언인 형태소만 선택하고, 품사가 용언일 때 종결어미 '다'를 결합합니다.
[token.form + '다' if token.tag in verb_tags else token.form 
 for token in tokens if token.tag in verb_tags + noun_tags]

### 형태소 분석 함수 생성

In [ ]:
# 문서를 입력받아 형태소 분석을 실행하고 리스트로 반환하는 함수를 생성합니다.
def tokenizer(text):
    tokens = kiwi.tokenize(text=text, typos='basic', stopwords=stopwords)
    return [token.form + '다' if token.tag in verb_tags else token.form
            for token in tokens if token.tag in verb_tags + noun_tags]

In [ ]:
# 형태소 분석기를 실행한 결과를 확인합니다.
tokenizer(text=corrected)

### BoW 생성: 문서 집합으로 형태소 분석 실행

In [ ]:
# 필요한 모듈을 임포트합니다.
from tqdm.notebook import tqdm

In [ ]:
# 형태소 분석 결과를 저장할 빈 리스트를 생성합니다.
morphs = []

In [ ]:
# 문서 집합에서 각 문서를 형태소로 나누고 일부 품사를 남긴 morphs를 생성합니다.
for document in tqdm(df['corrected']):
    tokens = tokenizer(text=document)
    morphs.append(tokens)

### BoW 확인: 사용자 사전에 추가할 단어 탐색

In [ ]:
# 한글 맞춤법 검사를 실행한 문서와 형태소 분석 결과를 비교합니다.
for i in range(5):
    print(f'{i}: {df["corrected"].iloc[i]}')
    print(f'{i}: {morphs[i]}', end='\n\n')

### 사용자 사전에 추가

In [ ]:
# 사용자 사전에 추가할 단어와 품사 목록을 각각 리스트로 생성합니다.
words = ['사무용', '저렴', '재구매', '사은 품', '구매 확정']
tags = ['NNG'] * 5

In [ ]:
# 반복문으로 사용자 사전에 새로운 단어를 추가합니다.
for word, tag in zip(words, tags):
    kiwi.add_user_word(word=word, tag=tag)

### BoW 생성: 사용자 사전 추가 후 형태소 분석 실행

In [ ]:
# 형태소 분석 결과를 저장할 빈 리스트를 생성합니다.
morphs = []

In [ ]:
# 사용자 사전을 적용하여 문서 집합으로 형태소 분석을 실행하고 morphs를 생성합니다.
for document in tqdm(df['corrected']):
    tokens = tokenizer(text=document)
    morphs.append(tokens)

In [ ]:
# morphs(형태소로 나눈 문서 집합)의 처음 5개 원소를 확인합니다.
[print(morph) for morph in morphs[:5]]

### [참고] 사용자 사전용 텍스트 파일 생성 및 읽기

### 바이그램 생성

In [ ]:
# 필요한 모듈을 임포트합니다.
from nltk import bigrams

In [ ]:
# morphs의 첫 번째 원소에서 연속하는 두 단어를 튜플로 묶은 bigram을 생성합니다.
bigram = bigrams(sequence=morphs[0])

In [ ]:
# bigram를 리스트로 변환한 결과를 확인합니다.
list(bigram)

In [ ]:
# morphs의 첫 번째 원소를 출력하고 위 코드 결과와 비교합니다.
print(morphs[0])

In [ ]:
# morphs의 전체 원소에서 생성한 bigram을 저장할 빈 리스트를 생성합니다.
bigram_list = []

In [ ]:
# morphs에서 연속하는 두 단어를 묶은 튜플을 bigram_list에 결합합니다.
for morph in morphs:
    bigram = bigrams(sequence=morph)
    bigram_list += list(bigram)

In [ ]:
# bigram_list를 시리즈로 변환하여 원소별 도수를 계산하고 처음 20개를 확인합니다.
pd.Series(data=bigram_list).value_counts().head(n=20)

### morphs 원소 전처리

In [ ]:
# morphs의 처음 5개 원소를 확인합니다.
morphs[:5]

In [ ]:
# 형태소(단어)에 있는 공백을 제거하고 morphs에 재할당합니다.
morphs = [[token.replace(' ', '') for token in morph] for morph in morphs]

In [ ]:
# morphs의 원소(리스트)를 하나의 문자열로 결합하고 corpus에 할당합니다.
corpus = [' '.join(morph) for morph in morphs]

In [ ]:
# corpus의 처음 5개 원소를 확인합니다.
corpus[:5]

### 문서-단어 행렬 생성

In [ ]:
# 필요한 모듈을 임포트합니다.
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# 문서별 단어 도수를 계산하는 객체를 생성합니다.
count_vectorizer = CountVectorizer(min_df=0.01)

In [ ]:
# 단어 도수를 성분으로 갖는 문서-단어 행렬을 생성합니다.
dtm = count_vectorizer.fit_transform(raw_documents=corpus).toarray()

### 문서-단어 행렬 확인

In [ ]:
# dtm을 데이터프레임으로 변환합니다.
dtm = pd.DataFrame(data=dtm, columns=count_vectorizer.get_feature_names_out())

In [ ]:
# dtm의 처음 10행을 확인합니다.
dtm.head(n=10)

In [ ]:
# dtm의 행 개수와 열 개수를 확인합니다.
dtm.shape

In [ ]:
# dtm의 단어별 도수 합계를 계산하고 상위 20개 단어를 확인합니다.
dtm.sum().sort_values(ascending=False).head(n=20)

## 문장 유사도

### 문서 벡터 간 코사인 거리 행렬 생성

In [ ]:
# 필요한 모듈을 임포트합니다.
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
# dtm의 행(문서) 벡터 간 코사인 거리를 2차원 배열로 반환합니다.
cos_dist = cosine_distances(X=dtm)

In [ ]:
# 코사인 거리 행렬의 주대각 성분을 무한대 값으로 변경합니다.
# [참고] np.fill_diagonal()는 실행 결과를 배열에 업데이트하므로 원본에 재할당하지 않습니다.
np.fill_diagonal(a=cos_dist, val=np.inf)

In [ ]:
# 코사인 거리 행렬을 반올림하면 결과를 확인하기 좋습니다.
cos_dist.round(3)

### 유사도 높은 문서 확인

In [ ]:
# 관심 있는 문서의 정수 인덱스를 설정합니다.
idx = 0

In [ ]:
# 관심 있는 문서의 코사인 거리 벡터에서 최솟값을 확인합니다.
# [참고] 배열 원소 중에 결측값이 있으면 최솟값 대신 nan을 반환합니다.
np.min(a=cos_dist[idx, :])

In [ ]:
# 관심 있는 문서의 코사인 거리 벡터 원소를 오름차순 정렬하고 정수 인덱스를 반환합니다.
# [참고] np.argsort 함수는 원소를 오름차순 정렬하고 인덱스를 1차원 배열로 반환합니다.
min_indices = np.argsort(a=cos_dist[idx, :])

In [ ]:
# 관심 있는 문서와 코사인 거리가 가장 가까운 5개 문서를 확인합니다.
df['corrected'].iloc[np.hstack(tup=(idx, min_indices[:5]))]

## End of Document